# Spatial dynamics analysis for `BioLeakyRNNTopo`

Diagnostic notebook testing whether the trained model actually uses **continuous
spatial structure** to solve the cue/delay/target task, or whether it collapses
to a 4-way categorical attractor regardless of the Gaussian RF input path.

Four analyses, each a different angle:
1. **Spatial activity maps** over pre-target delay (Amengual-style panels).
2. **Linear spatial decoder** `h(t) -> (tx, ty)`.
3. **Center-of-mass drift + spatial variance** of population activity.
4. **Spatial ablation** — silence neurons near target vs random vs far.

Uses `../checkpoints/stage2_topo.pt` as-is. Saves figures under `../figures/`.


In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from collections import defaultdict
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold

from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    filter_trials,
    decode_position_by_ctoa_bin,
    polynomial_regression,
)
from src.plotting import plot_decoding_vs_ctoa, plot_decoding_vs_ctoa_amengual

device = "cpu"
Path("figures").mkdir(exist_ok=True)


def make_model():
    return BioLeakyRNNTopo(
        input_size=7,
        hidden_size=180,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=0.10,
        rec_init="diag",
        use_ei=True,
        exc_ratio=0.80,
        use_dale=True,
        mask_seed=42,
        sheet_side=12,
        tau_ee=0.25,
        tau_ie=0.32,
        tau_ei=0.64,
        tau_ii=0.64,
        rf_sigma=0.3,
        tau_e_range=(50, 150),
        tau_i_range=(10, 50),
        use_adaptation=False,
        tau_adapt=100.0,
        g_adapt=0.5,
    ).to(device)


model = make_model()
ckpt = torch.load("checkpoints/stage2_topo.pt", weights_only=True)
model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()

H = int(model.hidden_size)
n_exc = int(model.n_exc)
coords = model.coords.detach().cpu().numpy()  # [H, 2]

STIM_POS = CuedTargetWithDistractorsV3.STIM_POS
CUE_POS = CuedTargetWithDistractorsV3.CUE_POS
DT = 20

print(f"Model loaded: H={H}, n_exc={n_exc}, rf_sigma={float(model.rf_sigma):.2f}")
print(f"coords range: [{coords.min():.2f}, {coords.max():.2f}]")

from src.figsave import autosave
autosave("05_spatial_analysis_topo")


## 1. Collect trials

No-distractor env for cleaner pre-target dynamics. We'll group by CTOA bin and
target location later.

In [ ]:
def make_env_eval():
    # no distractors — we want clean pre-target dynamics
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.0,
        distractor_strength=0.0,
        continuous_locations=True,
    )


torch.manual_seed(0)
np.random.seed(0)
trials = collect_trials(model, make_env_eval, n_trials=600, device=device)

outcomes = defaultdict(int)
for tr in trials:
    outcomes[tr["train_outcome"]] += 1
print("Outcomes:", dict(outcomes))

corr = filter_trials(trials, outcome="correct")
print(f"Correct: {len(corr)} / {len(trials)}  ({100*len(corr)/len(trials):.1f}%)")

bin_counts = defaultdict(int)
for tr in corr:
    bin_counts[tr["ctoa_bin"]] += 1
print("Correct-trial CTOA bin counts:", dict(bin_counts))

rep_bin = max(bin_counts, key=bin_counts.get)
rep_trials = [tr for tr in corr if tr["ctoa_bin"] == rep_bin]
print(f"Representative CTOA bin = {rep_bin},  n_trials = {len(rep_trials)}")

## 2. Spatial activity maps across pre-target window

Scatter all 180 neurons at their `model.coords` positions, colored by ReLU'd
activity. Gaussian KDE smoothing on a regular grid gives an Amengual-style
density map.

Rows = target locations (1..4). Columns = time snapshots relative to target
onset (`-400, -300, -200, -100, 0, +100 ms`). Red dot = target, yellow star = cue.

In [ ]:
def smooth_map(vals, pts, grid_size=80, sigma=0.12):
    """vals [H], pts [H,2] in [-1,+1]^2 (TOROIDAL) -> smooth [grid, grid].

    Gaussian smoothing uses wrapped distance (period 2.0) so the visualization
    is consistent with the model's toroidal RF and recurrent connectivity.
    """
    gx = np.linspace(-1, 1, grid_size)
    gy = np.linspace(-1, 1, grid_size)
    GX, GY = np.meshgrid(gx, gy)
    Z = np.zeros_like(GX)
    for i in range(len(vals)):
        dx = GX - pts[i, 0]; dx = dx - 2.0 * np.round(dx / 2.0)
        dy = GY - pts[i, 1]; dy = dy - 2.0 * np.round(dy / 2.0)
        d2 = dx ** 2 + dy ** 2
        Z += vals[i] * np.exp(-d2 / (2 * sigma ** 2))
    return Z


def theta_bin(tr, n_bins=8):
    """Continuous-task replacement for target_loc: angular bin 0..n_bins-1
    by cue_pos angle. 8 bins of 45 deg give ~12% of trials each — narrow
    enough that the mean activity inside a bin still shows a localized bump
    (4-quadrant binning of continuous trials washes the bump out)."""
    x, y = tr["cue_pos"]
    theta = np.arctan2(y, x)              # [-pi, pi]
    return int((theta + np.pi) / (2 * np.pi) * n_bins) % n_bins


N_BINS = 8
W_BEFORE = 25  # 500 ms pre-target at dt=20
W_AFTER = 10   # 200 ms post-target
snapshot_ts = np.array([-20, -15, -10, -5, 0, 5])
snapshot_ms = snapshot_ts * DT

# Pass 1: per-bin baseline-subtracted segment means.
# Baseline = per-trial mean of h over the pre-cue fixation window (h[:cue_onset]),
# subtracted from every timepoint BEFORE averaging across trials, so suppression
# below baseline is preserved as a signed delta.
bin_info = []   # per bin: (n_trials, tx, ty, cx, cy) or None when too few trials
bin_delta = []  # per bin: delta_h segment mean [W, H] or None
for ang_bin in range(N_BINS):
    bin_trials = [tr for tr in rep_trials if theta_bin(tr, N_BINS) == ang_bin]
    if len(bin_trials) < 3:
        bin_info.append(None); bin_delta.append(None); continue
    segs = []
    for tr in bin_trials:
        t0 = tr["target_onset"]
        h = tr["h"]
        s, e = t0 - W_BEFORE, t0 + W_AFTER + 1
        if s < 0 or e > len(h):
            continue
        h_base = h[:tr["cue_onset"]].mean(axis=0)   # pre-cue fixation baseline [H]
        segs.append(h[s:e] - h_base)
    if not segs:
        bin_info.append(None); bin_delta.append(None); continue
    delta = np.stack(segs).mean(axis=0)             # [W, H], signed
    tx, ty = np.mean([tr["target_pos"] for tr in bin_trials], axis=0)
    cx, cy = np.mean([tr["cue_pos"] for tr in bin_trials], axis=0)
    bin_info.append((len(bin_trials), tx, ty, cx, cy))
    bin_delta.append(delta)

# Smooth every snapshot map up front so the diverging color scale can be fixed
# symmetrically about 0 and shared across all subplots. Use the 99th percentile of
# pooled |Z| (not the max) so one late post-target peak does not wash out the gradient.
Z_maps = {}
for row in range(N_BINS):
    if bin_delta[row] is None:
        continue
    for col, ts in enumerate(snapshot_ts):
        Z_maps[(row, col)] = smooth_map(bin_delta[row][W_BEFORE + ts], coords, sigma=0.12)
_pool = np.concatenate([np.abs(Z).ravel() for Z in Z_maps.values()]) if Z_maps else np.array([1.0])
M = float(np.percentile(_pool, 99))
M = M if M > 0 else 1.0

# Pass 2: draw with fixed vmin=-M, vmax=+M on a diverging colormap.
fig, axes = plt.subplots(
    N_BINS, len(snapshot_ts), figsize=(2.3 * len(snapshot_ts), 2.0 * N_BINS),
    sharex=True, sharey=True,
)
im = None
for row in range(N_BINS):
    if bin_info[row] is None:
        for col in range(len(snapshot_ts)):
            axes[row, col].set_visible(False)
        continue
    n_trials, tx, ty, cx, cy = bin_info[row]
    theta_center_deg = (row + 0.5) * 360.0 / N_BINS - 180.0
    for col, (ts, ms) in enumerate(zip(snapshot_ts, snapshot_ms)):
        ax = axes[row, col]
        im = ax.imshow(
            Z_maps[(row, col)], origin="lower", extent=[-1, 1, -1, 1],
            cmap="RdBu_r", vmin=-M, vmax=M,
        )
        # neuron positions in black (white-centered diverging map hides white dots)
        ax.scatter(coords[:n_exc, 0], coords[:n_exc, 1], s=3, c="black", alpha=0.25)
        ax.scatter(coords[n_exc:, 0], coords[n_exc:, 1], s=6, c="black",
                   marker="^", alpha=0.4)
        ax.plot(tx, ty, "o", mfc="red", mec="white", mew=1.3, markersize=10)
        ax.plot(cx, cy, "*", mfc="yellow", mec="black", mew=0.6, markersize=11)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f"t = {int(ms):+d} ms", fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"theta ~ {theta_center_deg:+.0f}deg{chr(10)}(n={n_trials})",
                fontsize=9,
            )

if im is not None:
    fig.colorbar(im, ax=axes, fraction=0.015, pad=0.01,
                 label="delta h vs pre-cue baseline")
fig.suptitle(
    f"Spatial activity maps (baseline-subtracted, pre-cue) | CTOA bin {rep_bin} | "
    f"{N_BINS} angular bins | RdBu_r: blue=suppression, red=enhancement | "
    f"red=target, yellow=cue | color=99th pct",
    fontsize=12,
)
fig.savefig("figures/spatial_activity_map.png", dpi=140)
plt.show()


In [ ]:
# --- Growth views: does activity grow, and when? (reuses bin_delta from the cell above) ---
# bin_delta[row] is [W, H] signed delta, target-aligned (t=0 at index W_BEFORE).
rel_ms = (np.arange(-W_BEFORE, W_AFTER + 1)) * DT
valid = [r for r in range(N_BINS) if bin_delta[r] is not None]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.6))
mag_stack = []
for r in valid:
    mag = np.abs(bin_delta[r]).mean(axis=1)            # [W] mean over neurons
    theta_deg = (r + 0.5) * 360.0 / N_BINS - 180.0
    axL.plot(rel_ms, mag, lw=1, alpha=0.5, label=f"{theta_deg:+.0f}deg")
    mag_stack.append(mag)
mag_mean = np.stack(mag_stack).mean(0)
axL.plot(rel_ms, mag_mean, lw=3, color="k", label="mean across bins")
axL.axvline(0, ls="--", c="grey")
axL.set_xlabel("time rel. target onset (ms)"); axL.set_ylabel("mean |delta h| over neurons")
axL.set_title("Activity magnitude over time"); axL.legend(fontsize=7, ncol=2)

H_mat = np.stack([np.abs(bin_delta[r]).mean(axis=1) for r in valid])   # [n_valid, W]
im = axR.imshow(H_mat, aspect="auto", origin="lower", cmap="viridis",
                extent=[rel_ms[0], rel_ms[-1], 0, len(valid)])
axR.axvline(0, ls="--", c="w")
axR.set_yticks(np.arange(len(valid)) + 0.5)
axR.set_yticklabels([f"{(r + 0.5) * 360.0 / N_BINS - 180.0:+.0f}deg" for r in valid], fontsize=8)
axR.set_xlabel("time rel. target onset (ms)"); axR.set_title("Mean |delta h|: angular bin x time")
fig.colorbar(im, ax=axR, fraction=0.046, label="mean |delta h|")
fig.suptitle(f"Activity growth | CTOA bin {rep_bin}", fontsize=12)
fig.tight_layout()
fig.savefig("figures/spatial_activity_growth.png", dpi=140)
plt.show()


In [ ]:
# --- Per-timepoint normalized maps: rescale EACH time column to its own 99th-pct scale,
# so the spatial pattern is visible even when the magnitude is small. Cross-time magnitude
# is NOT comparable here (Cell 5 / the growth views above are for that). Reuses Z_maps. ---
col_M = []
for col in range(len(snapshot_ts)):
    vals = [np.abs(Z_maps[(r, col)]).ravel() for r in range(N_BINS) if (r, col) in Z_maps]
    cm = float(np.percentile(np.concatenate(vals), 99)) if vals else 1.0
    col_M.append(cm if cm > 0 else 1.0)

fig, axes = plt.subplots(
    N_BINS, len(snapshot_ts), figsize=(2.3 * len(snapshot_ts), 2.0 * N_BINS),
    sharex=True, sharey=True,
)
for row in range(N_BINS):
    if bin_info[row] is None:
        for col in range(len(snapshot_ts)):
            axes[row, col].set_visible(False)
        continue
    n_trials, tx, ty, cx, cy = bin_info[row]
    theta_center_deg = (row + 0.5) * 360.0 / N_BINS - 180.0
    for col, (ts, ms) in enumerate(zip(snapshot_ts, snapshot_ms)):
        ax = axes[row, col]
        ax.imshow(Z_maps[(row, col)], origin="lower", extent=[-1, 1, -1, 1],
                  cmap="RdBu_r", vmin=-col_M[col], vmax=col_M[col])
        ax.scatter(coords[:n_exc, 0], coords[:n_exc, 1], s=3, c="black", alpha=0.25)
        ax.plot(tx, ty, "o", mfc="red", mec="white", mew=1.3, markersize=10)
        ax.plot(cx, cy, "*", mfc="yellow", mec="black", mew=0.6, markersize=11)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f"t = {int(ms):+d} ms", fontsize=10)
        if col == 0:
            ax.set_ylabel(f"theta ~ {theta_center_deg:+.0f}deg", fontsize=9)
fig.suptitle(
    f"Per-timepoint normalized maps (each column scaled to its own 99th pct) | "
    f"CTOA bin {rep_bin} | red=target, yellow=cue | magnitude NOT comparable across time",
    fontsize=12,
)
fig.savefig("figures/spatial_activity_map_pernorm.png", dpi=140)
plt.show()


## 3. Linear decoder `h(t) -> (tx, ty)` — honest version

Previous version had a bug: "fixation baseline" used `target_onset - 600ms`,
which is INSIDE cue/delay for long-CTOA trials and contains full cue info.

Here we align by **cue onset** instead — this gives a clean pre-cue window
(guaranteed fixation epoch, since fixation duration ∈ [700, 1200] ms).

Decoder: Ridge(alpha=1.0), 5-fold CV, R² on continuous (tx, ty).
Baselines:
- **shuffled targets** — null, R² should hover around 0
- **pre-cue** — h at `cue_onset - 200ms`, deep in fixation → target not
  yet determined in h (should be R² ≈ 0 if properly pre-cue)

Interpretation:
- If R² jumps from 0 to high **at cue onset** → network encodes cue identity
  (categorical memory, not spatial drift)
- If R² rises gradually through delay → genuine spatial evolution

In [ ]:
def cv_r2(X, y, k=5, seed=0):
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    r2s = []
    for tr_ix, te_ix in kf.split(X):
        clf = Ridge(alpha=1.0)
        clf.fit(X[tr_ix], y[tr_ix])
        r2s.append(clf.score(X[te_ix], y[te_ix]))
    return float(np.mean(r2s)), float(np.std(r2s) / np.sqrt(k))


# Cue-onset alignment
# Cover from -400 ms (pre-cue fixation) through cue + 1500 ms delay
CB, CA = 20, 80  # -400 ms .. +1600 ms rel cue onset
cue_aligned_h, cue_aligned_xy = [], []
for tr in corr:
    t0 = tr["cue_onset"]
    h = tr["h"]
    T = len(h)
    s, e = t0 - CB, t0 + CA + 1
    if s < 0 or e > T:
        continue
    cue_aligned_h.append(h[s:e])
    tx, ty = tr["target_pos"]
    cue_aligned_xy.append([tx, ty])
cue_aligned_h = np.stack(cue_aligned_h, axis=0)
cue_aligned_xy = np.array(cue_aligned_xy)
cue_ms = np.arange(-CB, CA + 1) * DT
print(f"Cue-aligned h: {cue_aligned_h.shape}")

r2_cue = np.zeros(cue_aligned_h.shape[1])
sem_cue = np.zeros_like(r2_cue)
for t in range(cue_aligned_h.shape[1]):
    r2_cue[t], sem_cue[t] = cv_r2(cue_aligned_h[:, t, :], cue_aligned_xy)

rng = np.random.default_rng(0)
xy_shuf = cue_aligned_xy[rng.permutation(len(cue_aligned_xy))]
r2_shuf = np.zeros_like(r2_cue)
for t in range(cue_aligned_h.shape[1]):
    r2_shuf[t], _ = cv_r2(cue_aligned_h[:, t, :], xy_shuf)

# True pre-cue baseline: -200 ms rel cue onset (index = CB - 10)
pre_cue_idx = CB - 10
r2_precue, _ = cv_r2(cue_aligned_h[:, pre_cue_idx, :], cue_aligned_xy)
print(f"True pre-cue (t = -200 ms rel cue) R² = {r2_precue:+.3f}")

# Target-onset alignment
WB, WA = 30, 15
t_aligned_h, t_aligned_xy = [], []
for tr in corr:
    t0 = tr["target_onset"]
    h = tr["h"]
    T = len(h)
    s, e = t0 - WB, t0 + WA + 1
    if s < 0 or e > T:
        continue
    t_aligned_h.append(h[s:e])
    tx, ty = tr["target_pos"]
    t_aligned_xy.append([tx, ty])
t_aligned_h = np.stack(t_aligned_h, axis=0)
t_aligned_xy = np.array(t_aligned_xy)
tgt_ms = np.arange(-WB, WA + 1) * DT
r2_tgt = np.zeros(t_aligned_h.shape[1])
sem_tgt = np.zeros_like(r2_tgt)
for t in range(t_aligned_h.shape[1]):
    r2_tgt[t], sem_tgt[t] = cv_r2(t_aligned_h[:, t, :], t_aligned_xy)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

ax = axes[0]
ax.fill_between(cue_ms, r2_cue - sem_cue, r2_cue + sem_cue, alpha=0.25, color="C0")
ax.plot(cue_ms, r2_cue, "C0-", lw=2, label="h(t) -> (tx, ty)")
ax.plot(cue_ms, r2_shuf, "C1--", lw=1.5, label="shuffled")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="red", ls=":", label="cue onset")
ax.axvline(350, color="purple", ls=":", lw=0.8, label="cue offset")
ax.scatter(
    [-200], [r2_precue], color="k", s=60, zorder=5, label=f"pre-cue R² = {r2_precue:+.2f}"
)
ax.set_xlabel("time re cue onset (ms)")
ax.set_ylabel("R² (5-fold CV)")
ax.set_title("Cue-aligned: when does target info appear in h?")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
ax.fill_between(tgt_ms, r2_tgt - sem_tgt, r2_tgt + sem_tgt, alpha=0.25, color="C2")
ax.plot(tgt_ms, r2_tgt, "C2-", lw=2, label="h(t) -> (tx, ty)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="red", ls=":", label="target onset")
ax.set_xlabel("time re target onset (ms)")
ax.set_title("Target-aligned: pre-target holding")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)

fig.suptitle("Linear decoder R² over time (honest baselines)")
fig.tight_layout()
fig.savefig("figures/spatial_decoder.png", dpi=140)
plt.show()

# Geometric vs categorical test: decoder trained on 3 corners should predict
# the 4th well if the code is spatial/continuous; fails if it is discrete/one-hot.
print()
print("=== Leave-one-corner-out generalization test ===")
print("(If the code is geometric, decoder trained on 3 corners predicts")
print(" the 4th well. If it is discrete/categorical, it fails.)")

test_t_idx = CB + 20
X = cue_aligned_h[:, test_t_idx, :]
y = cue_aligned_xy
corner_of = []
for tr in corr:
    t0 = tr["cue_onset"]
    h = tr["h"]
    T = len(h)
    s, e = t0 - CB, t0 + CA + 1
    if s < 0 or e > T:
        continue
    corner_of.append(tr["target_loc"])
corner_of = np.array(corner_of)

for held_out in [1, 2, 3, 4]:
    tr_mask = corner_of != held_out
    te_mask = corner_of == held_out
    if te_mask.sum() == 0:
        continue
    clf = Ridge(alpha=1.0)
    clf.fit(X[tr_mask], y[tr_mask])
    y_te = y[te_mask]
    y_pred = clf.predict(X[te_mask])
    mean_pred = y_pred.mean(axis=0)
    mean_true = y_te.mean(axis=0)
    err = np.linalg.norm(mean_pred - mean_true)
    print(
        f"  held-out corner {held_out} (true={mean_true}): "
        f"mean pred = [{mean_pred[0]:+.3f}, {mean_pred[1]:+.3f}]  "
        f"err = {err:.3f}"
    )

## 4. Center of mass + spatial variance

Weighted COM across all 180 neurons: `com = Σᵢ relu(hᵢ) · rᵢ / Σᵢ relu(hᵢ)`.
Spatial variance: weighted mean squared distance from COM.

If the network is spatial, COM should move from cue location toward target, and
variance should drop as activity focuses onto target.

In [ ]:
def com_and_var(h_seq, coords):
    """h_seq [T, H], coords [H, 2] in [-1, +1]^2 (TOROIDAL) -> com [T, 2], var [T].

    COM via circular mean (mean of sin/cos per dim) - required because
    weighted mean of toroidal coordinates is ambiguous near the wrap.
    Spatial variance uses wrapped distance (period 2.0).
    """
    w = np.clip(h_seq, 0, None)
    s = w.sum(axis=1, keepdims=True)
    s = np.where(s > 0, s, 1.0)
    theta = coords * np.pi
    mean_sin = (w @ np.sin(theta)) / s
    mean_cos = (w @ np.cos(theta)) / s
    com = np.arctan2(mean_sin, mean_cos) / np.pi
    diff = coords[None, :, :] - com[:, None, :]
    diff = diff - 2.0 * np.round(diff / 2.0)
    d2 = (diff ** 2).sum(axis=-1)
    var = (w * d2).sum(axis=1) / s[:, 0]
    return com, var


rel_ms = np.arange(-WB, WA + 1) * DT

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_bins = plt.cm.hsv(np.linspace(0, 1, N_BINS + 1)[:-1])
var_curves = {}

for ang_bin in range(N_BINS):
    tr_set = [tr for tr in corr
              if theta_bin(tr, N_BINS) == ang_bin and tr["ctoa_bin"] == rep_bin]
    if len(tr_set) < 3:
        continue
    coms, vars_ = [], []
    for tr in tr_set:
        t0 = tr["target_onset"]
        h = tr["h"]
        s, e = t0 - WB, t0 + WA + 1
        if s < 0 or e > len(h):
            continue
        c, v = com_and_var(h[s:e], coords)
        coms.append(c); vars_.append(v)
    if not coms:
        continue
    com_m = np.mean(np.stack(coms), axis=0)
    var_m = np.mean(np.stack(vars_), axis=0)
    var_curves[ang_bin] = var_m

    color = cmap_bins[ang_bin]
    theta_deg = (ang_bin + 0.5) * 360.0 / N_BINS - 180.0

    ax = axes[0]
    ax.plot(com_m[:, 0], com_m[:, 1], color=color, lw=1.5, alpha=0.7, zorder=2,
            label=f"{theta_deg:+.0f}deg")
    tx, ty = np.mean([tr["target_pos"] for tr in tr_set], axis=0)
    cx, cy = np.mean([tr["cue_pos"] for tr in tr_set], axis=0)
    ax.plot(tx, ty, "s", color=color, mec="k", markersize=9, mew=0.8, zorder=4)
    ax.plot(cx, cy, "*", color=color, mec="k", markersize=11, mew=0.4, zorder=4)

axes[0].set_xlim(-1.1, 1.1); axes[0].set_ylim(-1.1, 1.1)
axes[0].axhline(0, color="k", lw=0.3); axes[0].axvline(0, color="k", lw=0.3)
axes[0].set_aspect("equal")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].set_title(f"COM trajectory by cue angle ({N_BINS} bins)\nsquare=target, star=cue")
axes[0].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5), title="theta")

ax = axes[1]
for ang_bin, vc in var_curves.items():
    theta_deg = (ang_bin + 0.5) * 360.0 / N_BINS - 180.0
    ax.plot(rel_ms, vc, color=cmap_bins[ang_bin], lw=2, label=f"{theta_deg:+.0f}deg")
ax.axvline(0, color="red", ls=":", label="target onset")
ax.set_xlabel("time rel target (ms)"); ax.set_ylabel("weighted spatial variance")
ax.set_title("Spatial variance over time")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig("figures/com_variance.png", dpi=140)
plt.show()


## 5. Spatial ablation

For each trial: silence k=20 neurons and measure hit rate. Compare:
- **baseline** (no ablation)
- **near target** — k nearest to `(tx, ty)`
- **far from target** — k farthest from `(tx, ty)`
- **random** — k random neurons

If the network uses space, *near-target* ablation should hurt most, *far* and
*random* less. If all are equally harmful, the task is solved by identity code,
not spatial code.

Ablation = multiply hidden state by mask after every `model.step()` so the
silenced neurons can't influence downstream dynamics.

In [ ]:
@torch.no_grad()
def rollout_on_reset_env(model, env, keep_mask):
    """env must already be .reset(). Returns outcome string."""
    x = env.ob.astype(np.float32)
    gt = env.gt.astype(np.int64)
    xb = torch.tensor(x, device=device).unsqueeze(0)  # [1, T, 7]
    h = model.init_hidden(1, xb.device)
    mask = torch.tensor(keep_mask, dtype=torch.float32, device=xb.device).view(1, -1)
    logits_list = []
    for t in range(xb.shape[1]):
        h, y = model.step(xb[:, t, :], h)
        h = h * mask
        logits_list.append(y)
    logits = torch.stack(logits_list, dim=1)[0].cpu().numpy()
    actions = np.argmax(logits, axis=-1)
    first = np.where(actions == 1)[0]
    if len(first) == 0:
        return "miss"
    t_first = int(first[0])
    return "correct" if gt[t_first] == 1 else "fa_or_abort"


K = 20


def mask_none(tx, ty, rng):
    return np.ones(H, dtype=np.float32)


def mask_near(tx, ty, rng):
    # Toroidal distance to target (period 2.0 in [-1, +1])
    diff = coords - np.array([tx, ty])
    diff = diff - 2.0 * np.round(diff / 2.0)
    d = np.linalg.norm(diff, axis=1)
    ix = np.argsort(d)[:K]
    m = np.ones(H, dtype=np.float32)
    m[ix] = 0.0
    return m


def mask_far(tx, ty, rng):
    # Toroidal distance to target (period 2.0 in [-1, +1])
    diff = coords - np.array([tx, ty])
    diff = diff - 2.0 * np.round(diff / 2.0)
    d = np.linalg.norm(diff, axis=1)
    ix = np.argsort(d)[-K:]
    m = np.ones(H, dtype=np.float32)
    m[ix] = 0.0
    return m


def mask_random(tx, ty, rng):
    ix = rng.choice(H, size=K, replace=False)
    m = np.ones(H, dtype=np.float32)
    m[ix] = 0.0
    return m


def run_ablation(mask_builder, n_trials=120, seeds=(0, 1, 2, 3, 4)):
    accs = []
    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        rng = np.random.default_rng(seed)
        hits = 0
        total = 0
        for _ in range(n_trials):
            env = make_env_eval()
            env.reset()
            tx, ty = env.trial["target_pos"]
            m = mask_builder(tx, ty, rng)
            outcome = rollout_on_reset_env(model, env, m)
            if outcome == "correct":
                hits += 1
            total += 1
        accs.append(hits / total)
    return float(np.mean(accs)), float(np.std(accs))


print("Running ablation (may take ~1-2 min)...")
results = {}
for name, fn in [
    ("baseline", mask_none),
    ("near target", mask_near),
    ("far target", mask_far),
    ("random", mask_random),
]:
    mu, sd = run_ablation(fn, n_trials=120, seeds=(0, 1, 2, 3, 4))
    results[name] = (mu, sd)
    print(f"  {name:15s}: hit = {100*mu:5.1f}% ± {100*sd:4.1f}%")

fig, ax = plt.subplots(figsize=(7, 4))
names = list(results.keys())
mus = [results[n][0] * 100 for n in names]
sds = [results[n][1] * 100 for n in names]
bar_colors = ["gray", "tomato", "steelblue", "orange"]
ax.bar(
    names, mus, yerr=sds, capsize=5, color=bar_colors, edgecolor="black", linewidth=0.5
)
ax.axhline(results["baseline"][0] * 100, color="k", ls="--", lw=0.8, label="baseline")
ax.set_ylabel("Hit rate (%)")
ax.set_title(f"Spatial ablation | k = {K} neurons silenced")
ax.set_ylim(0, 105)
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig("figures/ablation_bars.png", dpi=140)
plt.show()

## 6. Sanity: post-target map should peak at target

If this fails, the other analyses are meaningless. The combined activity map at
`target_onset + 100 ms` should clearly localize at the target position.

In [ ]:
fig, axes = plt.subplots(2, N_BINS // 2, figsize=(14, 7))
axes = axes.flatten()
for col, ang_bin in enumerate(range(N_BINS)):
    tr_set = [tr for tr in corr if theta_bin(tr, N_BINS) == ang_bin]
    theta_deg = (ang_bin + 0.5) * 360.0 / N_BINS - 180.0
    if len(tr_set) < 3:
        axes[col].set_title(f"theta {theta_deg:+.0f}deg: no data")
        continue
    h_sum = np.zeros(H); n = 0
    for tr in tr_set:
        t0 = tr["target_onset"]
        h = tr["h"]
        idx = t0 + 5
        if idx >= len(h):
            continue
        h_sum += np.clip(h[idx], 0, None); n += 1
    h_mean = h_sum / max(n, 1)
    Z = smooth_map(h_mean, coords, sigma=0.12)
    ax = axes[col]
    im = ax.imshow(Z, origin="lower", extent=[-1, 1, -1, 1], cmap="viridis")
    ax.scatter(coords[:, 0], coords[:, 1], s=3, c="white", alpha=0.3)
    tx, ty = np.mean([tr["target_pos"] for tr in tr_set], axis=0)
    ax.plot(tx, ty, "o", mfc="red", mec="white", mew=1.5, markersize=13)
    ax.set_title(f"theta {theta_deg:+.0f}deg (n={n})")
    ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Sanity check: post-target +100 ms activity, {N_BINS} angular bins")
fig.tight_layout()
fig.savefig("figures/sanity_post_target.png", dpi=140)
plt.show()


## Summary / verdict template

Fill in after running. Adjudicate each test:

| Test                               | Result       | Spatial? |
| ---------------------------------- | ------------ | -------- |
| 1. Activity maps show drift toward target | ?          | ?        |
| 2. Decoder R² rises to ~0.5+ near target  | ?          | ?        |
| 3. COM moves cue → target; variance drops | ?          | ?        |
| 4. Near-target ablation >> random/far     | ?          | ?        |
| **Sanity**: post-target peak at target    | **must be yes** | —    |

**Interpretation:**
- 4/4 yes → strong continuous spatial code
- 2–3 yes → partial spatial structure on top of discrete attractors
- 0–1 yes → network solves the task without using geometry (4-way categorical).
  In that case the Gaussian RF input is present but the recurrent dynamics
  collapsed it to corner-ID memory. To force spatial coding: add continuous
  target locations, or make `target_loc != cue_loc` with a geometric relation.


## 7. Position decoding vs CTOA (Amengual Fig 4A)

Decode the 4-quadrant target location from the mean hidden state, **per CTOA bin**,
in a pre-target window ($[-300, -100]$ ms) and a post-target window ($[100, 300]$ ms).
LDA + stratified 5-fold CV; chance = 0.25.

Amengual et al. Fig 4A find a **non-monotonic** (parabolic) profile: position
information builds up, peaks, then degrades with CTOA — the quadratic fit beats the
linear one (pre: quad R²=0.83 vs lin 0.36; post: quad R²=0.64 vs lin 0.57; AIC favours
quadratic in both).

This section is **self-contained**: it uses a separate model handle with
`noise_at_eval=True` (to break the trivial decoding ceiling) and a larger trial set
with distractors — leaving the noiseless model/trials used by sections 1–6 untouched.
Expectation for the topo model: decoding sits **high and roughly flat** across CTOA
(no quadratic advantage), i.e. it does **not** reproduce Amengual's parabola — the
parameter-free Gaussian RF input makes target position near-trivially decodable.

In [ ]:
# Noisy model handle + larger trial set (does not disturb sections 1-6).
model_dec = make_model()
model_dec.load_state_dict(ckpt["state_dict"], strict=False)
model_dec.eval()
model_dec.noise_at_eval = True


def make_env_decode():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
        continuous_locations=True,
    )


torch.manual_seed(0)
np.random.seed(0)
trials_dec = collect_trials(model_dec, make_env_decode, n_trials=4000, device=device)
corr_dec = filter_trials(trials_dec, outcome="correct")
print(f"Correct trials for decoding: {len(corr_dec)} / {len(trials_dec)}")

bin_counts = defaultdict(int)
for tr in corr_dec:
    bin_counts[tr["ctoa_bin"]] += 1
print("Correct-trial CTOA bin counts:", dict(sorted(bin_counts.items())))

In [ ]:
# pca_dims sweep: where (if anywhere) does CTOA modulate decoding?
dims_to_try = [None, 32, 16, 10, 6, 4]
for label, win in [("Pre-target (-300..-100)", (-300, -100)),
                   ("Post-target (100..300)", (100, 300))]:
    print(f"--- {label} ---")
    print(f'{"K":>5}  ' + "  ".join(f"b{i}" for i in range(10)))
    for k in dims_to_try:
        res = decode_position_by_ctoa_bin(
            corr_dec, window_ms=win, align_key="target_onset",
            outcome=None, min_trials=10, n_splits=5, pca_dims=k,
        )
        accs = "  ".join(f"{a:.2f}" for a in res["acc_per_bin"])
        print(f'{("all" if k is None else str(k)):>5}  {accs}')
    print()

In [ ]:
PCA_DIMS = 6  # adjust based on the sweep above

dec_pre = decode_position_by_ctoa_bin(
    corr_dec, window_ms=(-300, -100), align_key="target_onset",
    outcome=None, min_trials=10, n_splits=5, pca_dims=PCA_DIMS,
)
dec_post = decode_position_by_ctoa_bin(
    corr_dec, window_ms=(100, 300), align_key="target_onset",
    outcome=None, min_trials=10, n_splits=5, pca_dims=PCA_DIMS,
)

# Amengual Fig 4A style: both windows on one axis, colored by CTOA.
plot_decoding_vs_ctoa_amengual(dec_pre, dec_post)

In [ ]:
# Linear vs quadratic fit + AIC, against Amengual's targets.
print("=== Topo model: decoding vs CTOA regression ===")
for label, dec in [("Pre-target (-300..-100)", dec_pre),
                   ("Post-target (100..300)", dec_post)]:
    x = dec["ctoa_ms_mean"]
    y = dec["acc_per_bin"]
    n = len(x)
    print(f"\n{label}:")
    for deg in [1, 2]:
        reg = polynomial_regression(x, y, degree=deg)
        if reg["y_hat"] is not None:
            ss_res = np.sum((reg["y"] - reg["y_hat"]) ** 2)
            aic = n * np.log(ss_res / n + 1e-30) + 2 * (deg + 1)
            print(f'  deg-{deg}: R2={reg["r2"]:.3f}  p={reg["p_value"]:.4f}  AIC={aic:.2f}')
        else:
            print(f"  deg-{deg}: insufficient data")

print()
print("Amengual et al. Fig 4A targets:")
print("  Pre-target:  linear R2=0.36, quadratic R2=0.83 (AIC favours quadratic)")
print("  Post-target: linear R2=0.57, quadratic R2=0.64 (AIC favours quadratic)")

### Interpretation

Fill in after running with the full trial set, comparing against the printout above.

The key divergence from Amengual Fig 4A is one of **magnitude**, not just shape: the
topo model decodes target position at **near-ceiling accuracy (~0.85-0.97)** in
*every* CTOA bin, in both windows. Amengual's FEF, by contrast, shows position
information that is comparatively low and **gated by attention dynamics** over the
cue-target interval — it builds up, peaks, then degrades (the parabola).

Caveats when reading the regression:
- A significant quadratic fit here is often driven by the **sparse extreme-CTOA bins**
  (the highest bins have far fewer trials and noisier accuracy). Check `n_per_bin` and
  re-run with more trials before reading the curvature as a real build-up/degrade.
- Because absolute accuracy sits near the ceiling, the dynamic range is tiny, so R²/AIC
  comparisons are sensitive to a couple of edge bins.

Why the divergence: the parameter-free Gaussian receptive-field input injects target
position directly into the sheet, so position is near-trivially decodable regardless of
CTOA. The topo model lacks the attentional bottleneck that makes FEF position
information wax and wane. Present this as an explicit divergence from Amengual Fig 4A,
not a match.